# L&D Designs — Email Outreach Tool

Sends personalised cold outreach emails to leads from your spreadsheet using your Gmail account (free — no paid service needed).

---

## Step 1 — Enable 2-Factor Authentication on your Google Account

You need 2FA turned on before you can create an App Password.

1. Go to [myaccount.google.com](https://myaccount.google.com)
2. Click **Security** in the left sidebar
3. Under *How you sign in to Google*, click **2-Step Verification** and enable it if it isn't already

## Step 2 — Create a Gmail App Password

1. Stay on the **Security** page
2. Scroll down and click **App passwords** (you'll only see this once 2FA is on)
3. In the *App name* field type: `Lead Outreach`
4. Click **Create**
5. Google shows you a **16-character password** (4 groups of 4 letters separated by spaces, e.g. `abcd efgh ijkl mnop`)
6. **Copy it immediately** — it won't be shown again

## Step 3 — Fill in your credentials below and run the cell

> **Important:** Never share your App Password with anyone. It gives access to your Gmail account.

In [ ]:
# ── FILL IN YOUR DETAILS HERE ─────────────────────────────────────────────────
GMAIL_ADDRESS      = "your@gmail.com"
GMAIL_APP_PASSWORD = "xxxx xxxx xxxx xxxx"  # 16-char app password from Google

# Link to your pricing page (update this when you have a live URL)
PRICING_LINK = "https://htmlpreview.github.io/?https://github.com/YOUR_USERNAME/YOUR_REPO/blob/main/pricing.html"
# ──────────────────────────────────────────────────────────────────────────────

print("Settings saved.")
print("Gmail address  :", GMAIL_ADDRESS)
print("App password   :", "*" * len(GMAIL_APP_PASSWORD.replace(" ", "")))
print("Pricing link   :", PRICING_LINK)
print()
print("Run the next cell to upload your spreadsheet.")

---
## Cell 2 — Upload your leads spreadsheet

Your spreadsheet must have these column headers (exactly as shown):

| Column | Notes |
|---|---|
| `Business Name` | Name of the business |
| `Email Address` | Contact email |
| `Website Status` | Must be `NONE` (case-insensitive) to be included |

Leads are skipped if: they already have a website, the email is blank, or the email looks like it belongs to your own domain.

In [ ]:
# ── Cell 2: Upload spreadsheet & filter leads ─────────────────────────────────
from google.colab import files
import openpyxl

# Your own domain — emails containing this string will be skipped
OWN_DOMAIN = "lddesigns"

print("Select your leads .xlsx file...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print("Loaded file:", filename)

wb = openpyxl.load_workbook(filename)
ws = wb.active  # uses the first sheet

# Read headers from row 1
headers = [str(cell.value).strip() if cell.value is not None else "" for cell in ws[1]]
print("Columns found:", headers)

# Read all rows
all_leads = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if any(v is not None for v in row):
        all_leads.append(dict(zip(headers, row)))

print("Total rows loaded:", len(all_leads))

# ── Filter logic ──────────────────────────────────────────────────────────────
emailable_leads = []
skip_no_website = 0
skip_no_email   = 0
skip_own_domain = 0

for lead in all_leads:
    website_status = str(lead.get("Website Status") or "").strip().upper()
    email          = str(lead.get("Email Address") or "").strip()

    # Must have no website
    if website_status != "NONE":
        skip_no_website += 1
        continue

    # Must have an email address
    if not email or email.lower() == "none" or email.lower() == "nan":
        skip_no_email += 1
        continue

    # Skip if email looks like it belongs to our own domain (dedup)
    if OWN_DOMAIN.lower() in email.lower():
        skip_own_domain += 1
        continue

    emailable_leads.append(lead)

print()
print("--- Filter Results ---")
print("Skipped (has website or unknown status) :", skip_no_website)
print("Skipped (no email address)              :", skip_no_email)
print("Skipped (own domain)                    :", skip_own_domain)
print("Ready to email                          :", len(emailable_leads))
print()
if emailable_leads:
    print("First 5 leads to be emailed:")
    for lead in emailable_leads[:5]:
        bname = str(lead.get("Business Name") or "Unknown")
        email = str(lead.get("Email Address") or "")
        print(" -", bname, "|", email)
else:
    print("No emailable leads found. Check your column headers and Website Status values.")

---
## Cell 3 — Preview email & send

This cell previews the first email, then asks for confirmation before sending the full batch.

A **3–5 second random delay** is added between each send to reduce spam filter risk. Sending stops immediately if any email fails, and a summary is printed.

In [ ]:
# ── Cell 3: Preview & send emails ─────────────────────────────────────────────
import smtplib
import time
import random
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# ── Email template ────────────────────────────────────────────────────────────
def build_email(business_name, recipient_email):
    subject = "Quick question about " + business_name + "'s website"

    body = (
        "Hi,\n\n"
        "I noticed " + business_name + " doesn't have a website yet.\n\n"
        "I build websites for local businesses in the Wigan area — affordable prices, "
        "quick turnaround, and I handle everything from start to finish.\n\n"
        "If you're interested in a free quote, just reply to this email or "
        "WhatsApp me on 07301 181878.\n\n"
        "You can also see our pricing here: " + PRICING_LINK + "\n\n"
        "Thanks,\n"
        "Dylan\n"
        "L&D Designs"
    )

    msg = MIMEMultipart()
    msg["From"]    = GMAIL_ADDRESS
    msg["To"]      = recipient_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))
    return msg, subject, body


# ── Sanity checks ─────────────────────────────────────────────────────────────
if not emailable_leads:
    print("No leads to send to. Run Cell 2 first.")
elif GMAIL_ADDRESS == "your@gmail.com":
    print("Please fill in your Gmail address in Cell 1 first.")
elif GMAIL_APP_PASSWORD == "xxxx xxxx xxxx xxxx":
    print("Please fill in your Gmail App Password in Cell 1 first.")
else:
    # ── Preview first email ───────────────────────────────────────────────────
    first_lead   = emailable_leads[0]
    first_name   = str(first_lead.get("Business Name") or "Your Business")
    first_email  = str(first_lead.get("Email Address") or "")
    _, subj, body_preview = build_email(first_name, first_email)

    print("=" * 60)
    print("PREVIEW — first email that will be sent")
    print("=" * 60)
    print("To     :", first_email)
    print("Subject:", subj)
    print("-" * 60)
    print(body_preview)
    print("=" * 60)
    print()
    print("Total emails to send:", len(emailable_leads))
    print()

    # ── Confirmation ─────────────────────────────────────────────────────────
    confirm = input("Type YES to send all emails, or anything else to cancel: ").strip()

    if confirm != "YES":
        print("Cancelled. No emails were sent.")
    else:
        # ── Connect to Gmail SMTP ─────────────────────────────────────────────
        print()
        print("Connecting to Gmail SMTP...")
        try:
            server = smtplib.SMTP_SSL("smtp.gmail.com", 465)
            server.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
            print("Connected successfully.")
        except Exception as conn_err:
            print("Connection failed:", conn_err)
            print()
            print("Check your Gmail address and App Password in Cell 1.")
            raise

        # ── Send loop ─────────────────────────────────────────────────────────
        sent_log     = []   # list of dicts for the summary CSV
        sent_count   = 0
        failed_count = 0

        print()
        print("Sending...")
        print("-" * 50)

        for idx, lead in enumerate(emailable_leads):
            bname = str(lead.get("Business Name") or "Unknown")
            email = str(lead.get("Email Address") or "")
            ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            try:
                msg, _, _ = build_email(bname, email)
                server.sendmail(GMAIL_ADDRESS, email, msg.as_string())
                sent_count += 1
                status = "sent"
                print("[" + str(idx + 1) + "/" + str(len(emailable_leads)) + "] Sent   -> " + bname + " | " + email)
            except Exception as send_err:
                failed_count += 1
                status = "failed: " + str(send_err)
                print("[" + str(idx + 1) + "/" + str(len(emailable_leads)) + "] FAILED -> " + bname + " | " + email + " | " + str(send_err))
                sent_log.append({"business_name": bname, "email": email, "status": status, "timestamp": ts})
                print()
                print("Stopping batch due to send failure. Check the error above.")
                break

            sent_log.append({"business_name": bname, "email": email, "status": status, "timestamp": ts})

            # Delay between emails (skip delay after the last one)
            if idx < len(emailable_leads) - 1 and status == "sent":
                delay = random.uniform(3, 5)
                time.sleep(delay)

        server.quit()

        # ── Store log for Cell 4 ──────────────────────────────────────────────
        EMAIL_LOG = sent_log
        SENT_COUNT   = sent_count
        FAILED_COUNT = failed_count

        print("-" * 50)
        print()
        print("Done. Run Cell 4 to download your send log.")

---
## Cell 4 — Summary & download log

Prints a summary of the batch, saves a CSV log, and downloads it to your machine.

In [ ]:
# ── Cell 4: Summary & download log CSV ───────────────────────────────────────
import csv
from google.colab import files

# Safety check in case Cell 3 was skipped or didn't complete
try:
    EMAIL_LOG
except NameError:
    print("No log found. Run Cell 3 first.")
    raise SystemExit

# ── Print summary ─────────────────────────────────────────────────────────────
print("=" * 50)
print("SEND SUMMARY")
print("=" * 50)
print("Successfully sent :", SENT_COUNT)
print("Failed            :", FAILED_COUNT)
print("Total attempted   :", SENT_COUNT + FAILED_COUNT)
print()

if FAILED_COUNT > 0:
    print("Failed emails:")
    for entry in EMAIL_LOG:
        if entry["status"].startswith("failed"):
            print(" -", entry["business_name"], "|", entry["email"], "|", entry["status"])
    print()

# ── Save CSV log ──────────────────────────────────────────────────────────────
from datetime import datetime
log_filename = "email_log_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".csv"

with open(log_filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["business_name", "email", "status", "timestamp"])
    writer.writeheader()
    writer.writerows(EMAIL_LOG)

print("Log saved as:", log_filename)
print("Downloading...")
files.download(log_filename)
print("Check your Downloads folder for", log_filename)